# TCG Banlist Prediction

This notebook is for generating banlist predictions for TCG Format YuGiOh based on the most up to date data from YGOPRODECK. 

## Setup

In [38]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
import numpy as np
import plotnine as p9
import sklearn
import plotly.express as px

In [16]:
# Load data
df = pd.read_csv("../Data/combined_full_data.csv")
banlist = pd.read_csv("../Data/banlist_history_YGOPRO.csv")

We are going to need a way to join the banlist info to the other data, so here I check for whether the names show up the same. 

Note: I checked for the ids but I think the ids in banlist are coming from YGOPRODECK's id for the card. 

In [17]:
# Check if every card name in banlist shows up in df
banlist['card_name'].isin(df['name']).all()

np.True_

## Feature Engineering

### Generating features from the banlist data

In [18]:
# Keep only the TCG format from banlist
banlist = banlist[banlist['format'] == 'TCG']
banlist

,format,date,card_name,status,card_id,pretty_url,type
9954,TCG,1999-08-01,Dark Hole,Limited,53129443,dark-hole-4525,NaN
9955,TCG,1999-08-01,Raigeki,Limited,12580477,raigeki-1087,NaN
9956,TCG,1999-08-01,Trap Hole,Limited,4206964,trap-hole-379,NaN
9957,TCG,2000-04-01,Change of Heart,Limited,4031928,change-of-heart-359,NaN
9958,TCG,2000-04-01,Dark Hole,Limited,53129443,dark-hole-4525,NaN
...,...,...,...,...,...,...,...
20093,TCG,2025-04-07,Snake-Eye Ash,Semi-Limited,9674034,snake-eye-ash-14026,NaN
20094,TCG,2025-04-07,Snake-Eyes Poplar,Semi-Limited,90241276,snake-eyes-poplar-14182,NaN
20095,TCG,2025-04-07,Sword Ryzeal,Semi-Limited,35844557,sword-ryzeal-14643,NaN
20096,TCG,2025-04-07,Unchained Soul of Sharvara,Semi-Limited,41165831,unchained-soul-of-sharvara-13802,NaN


In [19]:
# For each card_name, count the number of times it's been on the banlist using pandas
ban_counts = banlist["card_name"].value_counts().reset_index()
ban_counts.columns = ["card_name", "num_banlists"]

# Join the counts with the original banlist
banlist = banlist.merge(ban_counts, on="card_name")
banlist

,format,date,card_name,status,card_id,pretty_url,type,num_banlists
0,TCG,1999-08-01,Dark Hole,Limited,53129443,dark-hole-4525,NaN,47
1,TCG,1999-08-01,Raigeki,Limited,12580477,raigeki-1087,NaN,57
2,TCG,1999-08-01,Trap Hole,Limited,4206964,trap-hole-379,NaN,1
3,TCG,2000-04-01,Change of Heart,Limited,4031928,change-of-heart-359,NaN,68
4,TCG,2000-04-01,Dark Hole,Limited,53129443,dark-hole-4525,NaN,47
...,...,...,...,...,...,...,...,...
10139,TCG,2025-04-07,Snake-Eye Ash,Semi-Limited,9674034,snake-eye-ash-14026,NaN,3
10140,TCG,2025-04-07,Snake-Eyes Poplar,Semi-Limited,90241276,snake-eyes-poplar-14182,NaN,3
10141,TCG,2025-04-07,Sword Ryzeal,Semi-Limited,35844557,sword-ryzeal-14643,NaN,1
10142,TCG,2025-04-07,Unchained Soul of Sharvara,Semi-Limited,41165831,unchained-soul-of-sharvara-13802,NaN,5


In [20]:
# For each card, separately find the number of times it's status has been Limited, Semi-Limited, or Forbidden
status_counts = banlist.groupby("card_name")["status"].value_counts().unstack(fill_value=0)
status_counts = status_counts.rename(columns={"Limited": "num_limited", "Semi-Limited": "num_semi_limited", "Forbidden": "num_forbidden"})

# Join status_counts with the original banlist
banlist = banlist.merge(status_counts, on="card_name", how="left")
banlist

,format,date,card_name,status,card_id,pretty_url,type,num_banlists,num_forbidden,num_limited,num_semi_limited
0,TCG,1999-08-01,Dark Hole,Limited,53129443,dark-hole-4525,NaN,47,11,28,8
1,TCG,1999-08-01,Raigeki,Limited,12580477,raigeki-1087,NaN,57,24,33,0
2,TCG,1999-08-01,Trap Hole,Limited,4206964,trap-hole-379,NaN,1,0,1,0
3,TCG,2000-04-01,Change of Heart,Limited,4031928,change-of-heart-359,NaN,68,51,17,0
4,TCG,2000-04-01,Dark Hole,Limited,53129443,dark-hole-4525,NaN,47,11,28,8
...,...,...,...,...,...,...,...,...,...,...,...
10139,TCG,2025-04-07,Snake-Eye Ash,Semi-Limited,9674034,snake-eye-ash-14026,NaN,3,0,2,1
10140,TCG,2025-04-07,Snake-Eyes Poplar,Semi-Limited,90241276,snake-eyes-poplar-14182,NaN,3,0,2,1
10141,TCG,2025-04-07,Sword Ryzeal,Semi-Limited,35844557,sword-ryzeal-14643,NaN,1,0,0,1
10142,TCG,2025-04-07,Unchained Soul of Sharvara,Semi-Limited,41165831,unchained-soul-of-sharvara-13802,NaN,5,0,4,1


In [ ]:
# Generate the number of times each card's status changed
# Sort banlist by card_name and date to track status changes over time
banlist_sorted = banlist.sort_values(['card_name', 'date'])

# For each card, identify status changes
status_changes = []

for card_name in banlist_sorted['card_name'].unique():
    card_data = banlist_sorted[banlist_sorted['card_name'] == card_name].copy()
    
    # Get all unique dates in the banlist to check for gaps
    all_dates = sorted(banlist['date'].unique())
    
    # Track previous status
    prev_status = None
    changes = 0
    
    for date in all_dates:
        # Check if card appears on this date
        current_entry = card_data[card_data['date'] == date]
        
        if len(current_entry) > 0:
            current_status = current_entry['status'].iloc[0]
        else:
            current_status = "Not Banned"  # Card not on banlist
        
        # Count status change
        if prev_status is not None and prev_status != current_status:
            changes += 1
        
        prev_status = current_status
    
    status_changes.append({'card_name': card_name, 'status_changes': changes})

# Convert to DataFrame
status_changes_df = pd.DataFrame(status_changes)

# Merge back with banlist to add this information
banlist = banlist.merge(status_changes_df, on='card_name', how='left')

banlist.head(5)

,card_name,status_changes
0,A Hero Lives,5
1,A-Assault Core,2
2,ABC-Dragon Buster,3
3,Abyss Dweller,1
4,Abyss Soldier,2


In [37]:
# Joining the banlist variables to df by card_name
# Rename card_name to name
banlist = banlist.rename(columns={'card_name': 'name'})

# Select relevant columns and remove duplicate rows
merge_list = banlist[['name', 'status_changes', 'num_banlists', 'num_forbidden', 'num_limited', 'num_semi_limited']].drop_duplicates()

# Join name, status_changes, and all the columns starting with 'num_'. Fill all NAs in the joined columns with 0
df = df.merge(merge_list, on='name', how='left').fillna(0)


In [25]:
# Get the date from time_pulled and make it its own column in df
df['time_pulled'] = pd.to_datetime(df['time_pulled'])
df['date_pulled'] = df['time_pulled'].dt.date
df

,card_id,name,desc,pend_desc,monster_desc,type,subtype,frame,race,archetype,...,price_range,price_stddev,is_tcg_exclusive,is_ocg_exclusive,days_since_tcg_release,days_since_ocg_release,first_market,market_delay,desc_length,date_pulled
0,80181649,"""A Case for K9""","When this card is activated: You can add 1 ""K9...",NaN,NaN,Spell Card,Continuous Spell,spell,Continuous,K9,...,0.00,NaN,0,0,30.0,161.0,OCG,131.0,92,2025-08-30
1,80181649,"""A Case for K9""","When this card is activated: You can add 1 ""K9...",NaN,NaN,Spell Card,Continuous Spell,spell,Continuous,K9,...,0.00,NaN,0,0,30.0,161.0,OCG,131.0,92,2025-08-30
2,34541863,"""A"" Cell Breeding Device","During each of your Standby Phases, put 1 A-Co...",NaN,NaN,Spell Card,Continuous Spell,spell,Continuous,Alien,...,24.31,12.000937,0,0,6681.0,6771.0,OCG,90.0,16,2025-08-30
3,64163367,"""A"" Cell Incubator",Each time an A-Counter(s) is removed from play...,NaN,NaN,Spell Card,Continuous Spell,spell,Continuous,Alien,...,1.02,0.477729,0,0,6499.0,6615.0,OCG,116.0,32,2025-08-30
4,91231901,"""A"" Cell Recombination Device",Target 1 face-up monster on the field; send 1 ...,NaN,NaN,Spell Card,Quick-Play Spell,spell,Quick-Play,Alien,...,0.77,0.320728,0,0,3222.0,3339.0,OCG,117.0,66,2025-08-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
509657,18865703,ZW - Ultimate Shield,When this card is Normal or Special Summoned: ...,NaN,NaN,Effect Monster,Effect Monster,effect,Aqua,Utopia,...,0.94,0.414116,0,0,4588.0,4657.0,OCG,69.0,56,2025-08-18
509658,76080032,ZW - Unicorn Spear,"You can target 1 ""Number C39: Utopia Ray"" you ...",NaN,NaN,Effect Monster,Effect Monster,effect,Beast,Utopia,...,0.83,0.373843,0,0,4965.0,5021.0,OCG,56.0,51,2025-08-18
509659,76080032,ZW - Unicorn Spear,"You can target 1 ""Number C39: Utopia Ray"" you ...",NaN,NaN,Effect Monster,Effect Monster,effect,Beast,Utopia,...,0.83,0.373843,0,0,4965.0,5021.0,OCG,56.0,51,2025-08-18
509660,76080032,ZW - Unicorn Spear,"You can target 1 ""Number C39: Utopia Ray"" you ...",NaN,NaN,Effect Monster,Effect Monster,effect,Beast,Utopia,...,0.83,0.373843,0,0,4965.0,5021.0,OCG,56.0,51,2025-08-18


Ideas to potentially implement: 
- Adding a "last_banlist_status" feature that - at any given date - includes the previous banlist status for a card

### Other Feature Engineering

#### Card Text Embeddings

In [ ]:
# Build the model input text per row
df['name'] = df['name'].fillna('').astype(str)
df['desc'] = df['desc'].fillna('').astype(str)
df['subtype'] = df['subtype'].fillna('').astype(str)

df['embed_text'] = (
    'Card Name: ' + df['name'] +
    ' | Description: ' + df['desc'] +
    ' | Type: ' + df['subtype']
)

df_embed = df[['card_id', 'name', 'embed_text']].copy()
df_embed = df_embed.drop_duplicates(subset=['embed_text'])
# df_embed

,card_id,name,embed_text
0,80181649,"""A Case for K9""","Card Name: ""A Case for K9"" | Description: When..."
2,34541863,"""A"" Cell Breeding Device","Card Name: ""A"" Cell Breeding Device | Descript..."
3,64163367,"""A"" Cell Incubator","Card Name: ""A"" Cell Incubator | Description: E..."
4,91231901,"""A"" Cell Recombination Device","Card Name: ""A"" Cell Recombination Device | Des..."
5,73262676,"""A"" Cell Scatter Burst","Card Name: ""A"" Cell Scatter Burst | Descriptio..."
...,...,...,...
491780,20938824,Maliss P March Hare,Card Name: Maliss P March Hare | Description: ...
491782,69272449,Maliss P White Rabbit,Card Name: Maliss P White Rabbit | Description...
491785,21848500,Maliss Q Hearts Crypter,Card Name: Maliss Q Hearts Crypter | Descripti...
491790,68059897,Maliss Q Red Ransom,Card Name: Maliss Q Red Ransom | Description: ...


In [49]:
# Instantiate the embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer("makiart/modernbert-base-ft-all-nli", device=device)

# Encode only unique texts for efficiency
unique_texts = df_embed['embed_text'].unique().tolist()
unique_embeddings = model.encode(
    unique_texts,
    batch_size=64,
    show_progress_bar=len(unique_texts) >= 256,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)

# Prepare embeddings as a DataFrame for merging back into df
dim = unique_embeddings.shape[1]
embed_cols = [f'emb_{i}' for i in range(dim)]
embed_df = pd.DataFrame(unique_embeddings, columns=embed_cols)
embed_df.insert(0, 'embed_text', unique_texts)

Batches:   0%|          | 0/210 [00:00<?, ?it/s]

In [52]:
# Reduce embedding dimensionality with PCA
from sklearn.decomposition import PCA

# Identify embedding columns
embed_cols = [c for c in embed_df.columns if c.startswith('emb_')]
print(f"Embedding dims: {len(embed_cols)}; rows: {len(embed_df)}")

# Fit PCA (use randomized solver for speed when n_components << n_features)
n_components = 64  # tune this later
pca = PCA(n_components=n_components, svd_solver='randomized', random_state=42)
emb_reduced = pca.fit_transform(embed_df[embed_cols].to_numpy())

Embedding dims: 768; rows: 13386


In [66]:
# Inspect variance explained
cum_var = pca.explained_variance_ratio_.cumsum()
print(f"Cumulative explained variance @ {n_components} comps: {cum_var[-1]:.3f}")

Cumulative explained variance @ 64 comps: 0.734


In [ ]:
# Replace original emb_* columns with reduced ones
emb_red_cols = [f"emb_pca_{i}" for i in range(n_components)]
emb_red_df = pd.DataFrame(emb_reduced, columns=emb_red_cols, index=embed_df.index)

# emb_red_df

embed_df_pca = pd.concat([embed_df.drop(columns=embed_cols), emb_red_df], axis=1)

# Join embed_df_pca with df_embed on embed_text
df_final = pd.merge(df_embed, embed_df_pca, on='embed_text', how='left')

# Join to df
df = pd.merge(df, df_final, on=['card_id', 'name', 'embed_text'], how='left')
df.drop(columns=['embed_text'], inplace=True)
# df

## Other Data Processing

In [89]:
# Create a unique_id for each card by rarity by combining card_id, rarity_code, and set_code into one string
df['unique_id'] = (
    df['card_id'].astype(str) + "_" +
    df['rarity_code'].astype(str) + "_" +
    df['set_code'].astype(str)
)

# Put unique_id as the first column
df = df[['unique_id'] + [col for col in df.columns if col != 'unique_id']]
df.head()

,unique_id,card_id,name,desc,pend_desc,monster_desc,type,subtype,frame,race,...,emb_pca_54,emb_pca_55,emb_pca_56,emb_pca_57,emb_pca_58,emb_pca_59,emb_pca_60,emb_pca_61,emb_pca_62,emb_pca_63
0,80181649_(StR)_JUSH-EN040,80181649,"""A Case for K9""","When this card is activated: You can add 1 ""K9...",0,0,Spell Card,Continuous Spell,spell,Continuous,...,0.023260,-0.039025,0.010795,0.022299,-0.038317,0.017704,0.021911,0.036427,0.033033,-0.028839
1,80181649_(SR)_JUSH-EN040,80181649,"""A Case for K9""","When this card is activated: You can add 1 ""K9...",0,0,Spell Card,Continuous Spell,spell,Continuous,...,0.023260,-0.039025,0.010795,0.022299,-0.038317,0.017704,0.021911,0.036427,0.033033,-0.028839
2,34541863_(C)_FOTB-EN043,34541863,"""A"" Cell Breeding Device","During each of your Standby Phases, put 1 A-Co...",0,0,Spell Card,Continuous Spell,spell,Continuous,...,-0.029645,0.059242,-0.006024,0.011794,-0.019094,0.016680,0.021843,0.044083,-0.023417,-0.025646
3,64163367_(C)_GLAS-EN062,64163367,"""A"" Cell Incubator",Each time an A-Counter(s) is removed from play...,0,0,Spell Card,Continuous Spell,spell,Continuous,...,0.015231,0.013163,0.076834,0.054834,0.005400,0.057889,-0.052287,0.009284,-0.050284,-0.011700
4,91231901_(C)_INOV-EN063,91231901,"""A"" Cell Recombination Device",Target 1 face-up monster on the field; send 1 ...,0,0,Spell Card,Quick-Play Spell,spell,Quick-Play,...,-0.004222,0.009369,-0.008056,0.009760,-0.008781,0.009610,-0.020797,-0.011339,-0.010503,-0.043284


#### Aggregate to Card-Over-Time Level

This is necessary because all rarities of a card get banned together.

In [105]:
rarity_level_features = [
    'price', 'tcgplayer_price', 'ebay_price', 'amazon_price',
    'coolstuffinc_price', 'views', 'viewsweek', 'upvotes',
    'downvotes'
]

# Aggregate rarity_level_features to card level instead by getting the mean, max, and min, standard deviation, and median
df_cards = df.groupby(['time_pulled', 'card_id']).agg({
    'price': ['mean', 'max', 'min', 'std', 'median'],
    'tcgplayer_price': ['mean', 'max', 'min', 'std', 'median'],
    'ebay_price': ['mean', 'max', 'min', 'std', 'median'],
    'amazon_price': ['mean', 'max', 'min', 'std', 'median'],
    'coolstuffinc_price': ['mean', 'max', 'min', 'std', 'median'],
    'views': ['mean', 'max', 'min', 'std', 'median'],
    'viewsweek': ['mean', 'max', 'min', 'std', 'median'],
    'upvotes': ['mean', 'max', 'min', 'std', 'median'],
    'downvotes': ['mean', 'max', 'min', 'std', 'median']
}).reset_index()
df_cards.columns = ['_'.join(col).strip() for col in df_cards.columns.values]

# Fix time_pulled_ and card_id_ columns
df_cards.rename(columns={
    'time_pulled_': 'time_pulled',
    'card_id_': 'card_id'
}, inplace=True)

df_cards


,time_pulled,card_id,price_mean,price_max,price_min,price_std,price_median,tcgplayer_price_mean,tcgplayer_price_max,tcgplayer_price_min,...,upvotes_mean,upvotes_max,upvotes_min,upvotes_std,upvotes_median,downvotes_mean,downvotes_max,downvotes_min,downvotes_std,downvotes_median
0,2025-08-18 23:47:23.400272,2511,0.00,0.00,0.00,0.00,0.00,0.09,0.09,0.09,...,0.0,0,0,0.0,0.0,0.0,0,0,0.0,0.0
1,2025-08-18 23:47:23.400272,10000,1961.53,1961.53,1961.53,NaN,1961.53,725.75,725.75,725.75,...,53.0,53,53,NaN,53.0,9.0,9,9,NaN,9.0
2,2025-08-18 23:47:23.400272,27551,0.31,1.24,0.00,0.62,0.00,0.10,0.10,0.10,...,10.0,10,10,0.0,10.0,8.0,8,8,0.0,8.0
3,2025-08-18 23:47:23.400272,32864,1.82,1.82,1.82,NaN,1.82,0.10,0.10,0.10,...,11.0,11,11,NaN,11.0,4.0,4,4,NaN,4.0
4,2025-08-18 23:47:23.400272,35699,0.00,0.00,0.00,NaN,0.00,0.21,0.21,0.21,...,5.0,5,5,NaN,5.0,2.0,2,2,NaN,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173870,2025-08-30 20:13:11.046288,501000002,0.00,0.00,0.00,NaN,0.00,0.00,0.00,0.00,...,7.0,7,7,NaN,7.0,0.0,0,0,NaN,0.0
173871,2025-08-30 20:13:11.046288,501000003,0.00,0.00,0.00,NaN,0.00,0.00,0.00,0.00,...,6.0,6,6,NaN,6.0,1.0,1,1,NaN,1.0
173872,2025-08-30 20:13:11.046288,501000004,0.00,0.00,0.00,NaN,0.00,0.00,0.00,0.00,...,6.0,6,6,NaN,6.0,0.0,0,0,NaN,0.0
173873,2025-08-30 20:13:11.046288,501000006,0.00,0.00,0.00,NaN,0.00,600.00,600.00,600.00,...,4.0,4,4,NaN,4.0,1.0,1,1,NaN,1.0


In [108]:
# Define regular feature columns
regular_features = [
    "banlist_tcg", 'time_pulled', 'card_id', 'subtype', 'race', 'archetype', 'atk', 'def', 'level', 
    'attribute', 'linkval', 'linkmarkers', 'scale',
    'has_effect', 'num_printings', 'num_rarities', 'num_formats', 
    'is_tcg_exclusive', 'is_ocg_exclusive', 'days_since_tcg_release', 'days_since_ocg_release', 'first_market', 'market_delay', 'desc_length',
    'status_changes', 'num_banlists', 'num_forbidden', 'num_limited', 'num_semi_limited'
]

# Generate PCA embedding columns
emb_pca_cols = [f'emb_pca_{i}' for i in range(64)]

# Combine all columns
model_columns = regular_features + emb_pca_cols

In [109]:
df_model = df[model_columns].copy()
df_model.drop_duplicates(inplace=True)
df_merged = pd.merge(df_model, df_cards, on=['time_pulled', 'card_id'], how='left')

In [110]:
df_merged

,banlist_tcg,time_pulled,card_id,subtype,race,archetype,atk,def,level,attribute,...,upvotes_mean,upvotes_max,upvotes_min,upvotes_std,upvotes_median,downvotes_mean,downvotes_max,downvotes_min,downvotes_std,downvotes_median
0,Not Banned,2025-08-30 20:13:11.046288,80181649,Continuous Spell,Continuous,K9,0.0,0.0,0.0,Spell,...,0.0,0,0,0.0,0.0,0.0,0,0,0.0,0.0
1,Not Banned,2025-08-30 20:13:11.046288,34541863,Continuous Spell,Continuous,Alien,0.0,0.0,0.0,Spell,...,118.0,118,118,NaN,118.0,110.0,110,110,NaN,110.0
2,Not Banned,2025-08-30 20:13:11.046288,64163367,Continuous Spell,Continuous,Alien,0.0,0.0,0.0,Spell,...,25.0,25,25,NaN,25.0,20.0,20,20,NaN,20.0
3,Not Banned,2025-08-30 20:13:11.046288,91231901,Quick-Play Spell,Quick-Play,Alien,0.0,0.0,0.0,Spell,...,19.0,19,19,NaN,19.0,15.0,15,15,NaN,15.0
4,Not Banned,2025-08-30 20:13:11.046288,73262676,Quick-Play Spell,Quick-Play,Alien,0.0,0.0,0.0,Spell,...,12.0,12,12,NaN,12.0,4.0,4,4,NaN,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173870,Not Banned,2025-08-18 23:47:23.400272,2648201,Effect Monster,Beast,Utopia,1000.0,1000.0,4.0,LIGHT,...,6.0,6,6,NaN,6.0,0.0,0,0,NaN,0.0
173871,Not Banned,2025-08-18 23:47:23.400272,95886782,Effect Monster,Beast,Zexal,800.0,1600.0,4.0,LIGHT,...,3.0,3,3,NaN,3.0,0.0,0,0,NaN,0.0
173872,Not Banned,2025-08-18 23:47:23.400272,81471108,Effect Monster,Dragon,Utopia,1300.0,1800.0,5.0,WIND,...,6.0,6,6,0.0,6.0,1.0,1,1,0.0,1.0
173873,Not Banned,2025-08-18 23:47:23.400272,18865703,Effect Monster,Aqua,Utopia,0.0,2000.0,4.0,EARTH,...,4.0,4,4,0.0,4.0,11.0,11,11,0.0,11.0


# Build Model

## Model Preprocessing

In [113]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [117]:
# Convert select numeric features to categorical
categorical_features = ['has_effect', 'is_tcg_exclusive', 'is_ocg_exclusive', 'card_id']
df_merged[categorical_features] = df_merged[categorical_features].astype('category')

In [128]:
# Handle categorical variables
categorical_cols = ['subtype', 'race', 'archetype', 'attribute', 'linkmarkers', 
                    'first_market', 'has_effect', 'is_tcg_exclusive', 'is_ocg_exclusive']
label_encoders = {}

df_model_prep = df_merged.copy()

# Encode categorical variables
for col in categorical_cols:
    if col in df_model_prep.columns:
        # If column is categorical, add 'Unknown' to categories first
        if isinstance(df_model_prep[col].dtype, pd.CategoricalDtype):
            df_model_prep[col] = df_model_prep[col].cat.add_categories(['Unknown'])
        df_model_prep[col] = df_model_prep[col].fillna('Unknown')
        le = LabelEncoder()
        df_model_prep[col] = le.fit_transform(df_model_prep[col].astype(str))
        label_encoders[col] = le

# Fill remaining NaN values for all columns besides banlist_tcg, time_pulled, and card_id (but keep the columns in)
exclude_cols = ['banlist_tcg', 'time_pulled', 'card_id']
fill_cols = [col for col in df_model_prep.columns if col not in exclude_cols]
df_model_prep[fill_cols] = df_model_prep[fill_cols].fillna(0)

# Convert time_pulled to numeric
df_model_prep['time_pulled'] = pd.to_numeric(df_model_prep['time_pulled'], errors='coerce')

In [129]:
# Prepare features and target
X = df_model_prep.drop(['card_id', 'banlist_tcg'], axis=1)
y = df_model_prep['banlist_tcg']

# Encode target variable
le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y.fillna('Not Listed'))

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

In [130]:
# Create LightGBM datasets
train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

# Set parameters
params = {
    'objective': 'multiclass',
    'num_class': len(le_target.classes_),
    'metric': 'multi_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.01,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0
}

# Train the model
model_lgb = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, valid_data],
    valid_names=['train', 'eval'],
    num_boost_round=1000,
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 50 rounds
[100]	train's multi_logloss: 0.00746624	eval's multi_logloss: 0.00750534
[200]	train's multi_logloss: 0.00161905	eval's multi_logloss: 0.00163311
[300]	train's multi_logloss: 0.000365296	eval's multi_logloss: 0.000369046
[400]	train's multi_logloss: 8.36368e-05	eval's multi_logloss: 8.46985e-05
[500]	train's multi_logloss: 1.93002e-05	eval's multi_logloss: 1.95904e-05
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

In [131]:
# Make predictions
y_pred = model_lgb.predict(X_test, num_iteration=model_lgb.best_iteration)
y_pred_classes = y_pred.argmax(axis=1)

In [132]:
# Print results
print("Classification Report:")
print(classification_report(y_test, y_pred_classes, target_names=le_target.classes_))

print("\nFeature Importance (top 20):")
feature_importance = model_lgb.feature_importance(importance_type='gain')
feature_names = X.columns
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False)
print(importance_df.head(20))

Classification Report:
              precision    recall  f1-score   support

   Forbidden       1.00      1.00      1.00       283
     Limited       1.00      1.00      1.00       213
  Not Banned       1.00      1.00      1.00     34237
Semi-Limited       1.00      1.00      1.00        42

    accuracy                           1.00     34775
   macro avg       1.00      1.00      1.00     34775
weighted avg       1.00      1.00      1.00     34775


Feature Importance (top 20):
                    feature     importance
22           status_changes  381978.818162
24            num_forbidden  281226.307818
25              num_limited  142940.617352
23             num_banlists   58114.042996
17   days_since_tcg_release   55410.582667
26         num_semi_limited   40117.409992
96     tcgplayer_price_mean    5983.130262
12            num_printings    5041.761918
76               emb_pca_49    4381.901092
44               emb_pca_17    4312.314083
83               emb_pca_56    3647.641